In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
%matplotlib inline

In [ ]:
import kagglehub
from sklearn.ensemble import RandomForestRegressor
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()
check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop('Order_ID', axis = 1)
df.head(1)

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)
#with weather I will use the mode or most repeated value
df['Weather'] = df['Weather'].fillna(df['Weather'].mode()[0])
#Same logic with Traffic Level, Time of day since they are labels:
df['Traffic_Level'] = df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0])
df['Time_of_Day'] = df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0])
#with delevery time and experience yrs i will take the average since they are numbers:
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())
df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].mean())
check_missing_values(df)


In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
le = LabelEncoder()
for col in categorical_cols:
  df[col] = le.fit_transform(df[col])
df.head()

In [ ]:
# Task 5: Write your code here:
numerical_cols = df.select_dtypes(include=["number"]).columns.drop("Delivery_Time")

scaler = StandardScaler()

# scale the `numerical_cols`
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df.head()

In [ ]:
# Task 6: Write your code here:
#Answer: this is not needed we are not making classification

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
n_splits = 5  # K=5 Folds
model = RandomForestRegressor(n_estimators=200)
# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
lr_mae = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)): #the .split() function returns the idex of the test and train
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train, y_train)
  # Validate
  y_pred = model.predict(X_test)

  # Calculate evaluation metrics
  mae = mean_absolute_error(y_test,y_pred)

  # Store results
  lr_mae.append(mae)
print("The mean MAE across the folds:",np.mean(lr_mae))

In [ ]:
# Task 1: Write your code here:
coeffs = model.coef_
print(coeffs)

In [ ]:
# Task 2: Write your code here:
plt.hist(y_pred, bins = 30, color = 'red', edgecolor = 'black')
plt.grid(True)
plt.xlabel("Predicted Delivery Time")
plt.ylabel("Frequency of predicted Delievery Time")
plt.title("Distribution of Predicted Delivery Time")
plt.show()

In [ ]:
# Task Bonus: Write your code here: